In [0]:
%pip install --upgrade numpy pandas pyarrow db-dtypes google-cloud-bigquery google-auth
dbutils.library.restartPython()

# 01 — Criação das Origens de Dados

Este notebook simula as fontes de dados do **Indicador Criança Alfabetizada** (INEP / Base dos Dados).

São criadas três origens com padrões distintos de ingestão:
1. **Dados estruturados**: UF, municípios e metas nacionais (padrão API)
2. **CDC (Change Data Capture)**: Atualizações de metas por UF ao longo do tempo
3. **Arquivos JSON**: Microdados do indicador por município (ingestão de arquivos)

> **Nota**: Em produção, os dados seriam obtidos diretamente da plataforma
> [Base dos Dados](https://basedosdados.org/) via biblioteca `basedosdados` + BigQuery.
> Esta simulação usa `spark.createDataFrame()` para compatibilidade com Databricks Serverless.

**Próximo passo**: executar `02_carga_camada_bronze.py`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
import json
import os

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IngesterDados") \
    .getOrCreate()


In [0]:
FILTRO_ANO_INICIAL = 2023
FILTRO_ANO_FINAL = 2024

## 1. Origem Estruturada: Dimensão UF e Metas Nacionais

Simula resposta de API com as 27 Unidades Federativas e metas anuais do programa
**Compromisso Nacional Criança Alfabetizada** (meta: 100% até 2030).

In [0]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Workspace/1AIST_projects/tech-challenge-02/notebooks/aist-tech-challenge02-30d3d92fb7ed.json"

In [0]:
with open(os.environ["GOOGLE_APPLICATION_CREDENTIALS"]) as f:
    chave = json.load(f)

In [0]:
from google.cloud import bigquery
import os
from google.oauth2 import service_account

# Cria as credenciais
credentials = service_account.Credentials.from_service_account_info(chave)
# Aponta para o arquivo de credenciais
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 

client = bigquery.Client(credentials=credentials, project="aist-tech-challenge02")



In [0]:
import json
import base64

# 'chave' já é o dict que você usou para criar as credenciais
chave_json = json.dumps(chave)
sa_b64 = base64.b64encode(chave_json.encode("utf-8")).decode("utf-8")

In [0]:
import os
print(os.path.exists("/Workspace/1AIST_projects/tech-challenge-02/notebooks/aist-tech-challenge02-30d3d92fb7ed.json"))

In [0]:
def read_bq_table(table_name: str):
    return (
        spark.read.format("bigquery")
        .option("table", f"basedosdados.br_inep_avaliacao_alfabetizacao.{table_name}")
        .option("parentProject", "aist-tech-challenge02")
        .option("credentials", sa_b64)
        .load()
    )

In [0]:
print("Iniciando carregamento de uf:")
df_uf = read_bq_table("uf")
print("Iniciando carregamento de municipio:")
df_municipio = read_bq_table("municipio")
print("Iniciando carregamento de dicionario:")
df_dicionario = read_bq_table("dicionario")
print("Iniciando carregamento de alunos:")
df_alunos = read_bq_table("alunos")
print("Iniciando carregamento de meta_alfabetizacao_municipio:")
df_meta_alf_mun = read_bq_table("meta_alfabetizacao_municipio")
print("Iniciando carregamento de meta_alfabetizacao_uf:")
df_meta_alf_uf = read_bq_table("meta_alfabetizacao_uf")
print("Iniciando carregamento de meta_alfabetizacao_brasil:")
df_meta_alf_br = read_bq_table("meta_alfabetizacao_brasil")

In [0]:
df_alunos.printSchema()

## 4. Persistência das Origens no Schema `origens`

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS origens")

tabelas_origem = {
    "origens.tc02_uf": df_uf
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_meta_brasil": df_meta_alf_br
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_meta_uf": df_meta_alf_uf
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_meta_mun": df_meta_alf_mun
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_alunos": df_alunos
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_municipio": df_municipio
        .withColumn("_data_criacao_origem", F.current_timestamp()),

    "origens.tc02_dicionario": df_dicionario
        .withColumn("_data_criacao_origem", F.current_timestamp()),
}

for nome_tabela, df in tabelas_origem.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

print("Tabelas de origem criadas com sucesso:")
for nome_tabela in tabelas_origem:
    print(f"  - {nome_tabela} => {spark.read.table(nome_tabela).count()} linhas")

print("\nPróximo passo: executar 02_carga_camada_bronze.py")

In [0]:
df_municipio.count()

In [0]:
df_alunos.show(3, truncate=False)

In [0]:
df_alunos.printSchema()

In [0]:
df_alunos.count()

In [0]:
pd_alunos = df_alunos.toPandas()

In [0]:
pd_alunos['id_aluno'].max()